In [3]:
--Fact table
CREATE OR REPLACE TABLE fact_monthly_property_stats AS
SELECT
    date_trunc('month', transfer_date) AS fact_month,
    county,
    town,
    property_type,
    old_new,
    duration,
    (case
        when ppd_category_type = "A" then "Standard Sale"
        when ppd_category_type = "B" then "Repossession and Other"
        else "Other"
    end) as transaction_category,
    count(transaction_id) as total_sales,
    AVG(price) as avg_price,
    percentile(price, 0.5) AS median_price,
    sum(price) AS total_sales_amount
from lk_property_price_bronze.dbo.property_prices_silver
group by
    date_trunc('month', transfer_date),
    county,
    town,
    property_type,
    old_new,
    duration,
    (case
        when ppd_category_type = "A" then "Standard Sale"
        when ppd_category_type = "B" then "Repossession and Other"
        else "Other"
    end);


StatementMeta(, 1f31a6fc-a1fa-4ad7-9d06-e8ddfd2b7019, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
--Dimentional Date table
CREATE OR REPLACE TABLE dim_date as
SELECT DISTINCT
    date_trunc('month', transfer_date) as date_key,
    year(transfer_date) as year,
    MONTH(transfer_date) as month_num,
    date_format(transfer_date, 'MMMM') as month_name,
    concat('Q', quarter(transfer_date),' ', year(transfer_date)) as quarter_year
from lk_property_price_bronze.dbo.property_prices_silver;

StatementMeta(, 1f31a6fc-a1fa-4ad7-9d06-e8ddfd2b7019, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
--Dimentional Property Type Mapping Table
CREATE OR REPLACE TABLE dim_property_type as
select 
    'D' as type_code,
    'Detached' as  type_name,
    'Houses' as category
UNION ALL
select 'S', 'Semi-Detached', 'Houses'
UNION ALL
select 'T', 'Terraced', 'Houses'
UNION ALL
select 'F', 'Flat', 'Flats'
UNION ALL
select 'O', 'Other', 'Other';

StatementMeta(, 1f31a6fc-a1fa-4ad7-9d06-e8ddfd2b7019, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
-- Tenure (Freehold vs Leasehold)
CREATE OR REPLACE TABLE dim_tenure as
select 'F' AS tenure_code, 'Freehold' as tenure_description 
UNION ALL
select 'L', 'Leasehold';

-- Build Type (New Build vs Established)
CREATE OR REPLACE TABLE dim_build_type AS
select 'Y' as build_code, 'New Build' AS build_description 
UNION ALL
select 'N', 'Existing';

StatementMeta(, 1f31a6fc-a1fa-4ad7-9d06-e8ddfd2b7019, 9, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
CREATE OR REPLACE TABLE dim_geography AS
SELECT DISTINCT
    county,
    town
FROM dbo.property_prices_silver;

StatementMeta(, 1f31a6fc-a1fa-4ad7-9d06-e8ddfd2b7019, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>